In [20]:
# ===================== 0. IMPORT THƯ VIỆN =====================
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms, datasets
from efficientnet_pytorch import EfficientNet

from PIL import Image
import numpy as np

# ===================== 1. CẤU HÌNH CHUNG =====================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Đường dẫn dữ liệu (để lấy class_names cho đúng 22 lớp)
train_data_dir = r"E:\archive\SkinDisease\SkinDisease\train"

# Đường dẫn checkpoint 3 mô hình (SỬA LẠI CHO ĐÚNG)
VGG_CKPT_PATH   = r"E:\archive\PiplineCode pretrain=True\vgg16_skin_best.pth"
RES50_CKPT_PATH = r"E:\archive\PiplineCode pretrain=True\resnet50_pretrained_fulloption.pth"
EFFB4_CKPT_PATH = r"E:\archive\PiplineCode pretrain=True\efficientnet_b4_pretrained_fulloption.pth"

IMAGE_SIZE   = 300
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# ===================== 2. LẤY DANH SÁCH CLASS TỪ THƯ MỤC TRAIN =====================
# Sử dụng ImageFolder để đảm bảo mapping class_index giống lúc train
tmp_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor()
])
train_ds = datasets.ImageFolder(train_data_dir, transform=tmp_transform)

class_names = train_ds.classes
num_classes = len(class_names)

print(f"📂 Số lớp (lấy từ folder train): {num_classes}")
print("Danh sách lớp:", class_names)

# Transform dùng cho inference
inference_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# ===================== 3. HÀM KHỞI TẠO & LOAD 3 MODEL =====================
def load_vgg16(num_classes, ckpt_path):
    """
    Khởi tạo VGG16 với num_classes đầu ra, rồi load state_dict từ checkpoint.
    """
    try:
        weights = models.VGG16_Weights.IMAGENET1K_V1
        model = models.vgg16(weights=weights)
    except AttributeError:  # phiên bản torchvision cũ
        model = models.vgg16(pretrained=True)

    in_features = model.classifier[6].in_features
    model.classifier[6] = nn.Linear(in_features, num_classes)

    state = torch.load(ckpt_path, map_location="cpu")
    model.load_state_dict(state)
    model.to(device)
    model.eval()
    print("✅ VGG16 loaded.")
    return model

def load_resnet50(num_classes, ckpt_path):
    """
    Khởi tạo ResNet50 với num_classes đầu ra, rồi load state_dict.
    """
    try:
        weights = models.ResNet50_Weights.IMAGENET1K_V1
        model = models.resnet50(weights=weights)
    except AttributeError:
        model = models.resnet50(pretrained=True)

    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)

    state = torch.load(ckpt_path, map_location="cpu")
    model.load_state_dict(state)
    model.to(device)
    model.eval()
    print("✅ ResNet50 loaded.")
    return model

def load_efficientnet_b4(num_classes, ckpt_path):
    """
    Khởi tạo EfficientNet-B4 với num_classes đầu ra, rồi load state_dict.
    """
    model = EfficientNet.from_pretrained('efficientnet-b4', num_classes=num_classes)
    state = torch.load(ckpt_path, map_location="cpu")
    model.load_state_dict(state)
    model.to(device)
    model.eval()
    print("✅ EfficientNet-B4 loaded.")
    return model

# ===================== 4. KHỞI TẠO 3 MÔ HÌNH (22 LỚP) =====================
vgg16_model    = load_vgg16(num_classes, VGG_CKPT_PATH)
resnet50_model = load_resnet50(num_classes, RES50_CKPT_PATH)
effb4_model    = load_efficientnet_b4(num_classes, EFFB4_CKPT_PATH)

# ===================== 5. HÀM DỰ ĐOÁN CHO 1 MÔ HÌNH =====================
def predict_single_model(model, img_tensor):
    """
    img_tensor: [1, 3, H, W] (đã transform + to(device))
    return: (pred_idx, pred_prob, prob_vec_numpy)
    """
    with torch.no_grad():
        outputs = model(img_tensor)           # [1, num_classes]
        probs   = F.softmax(outputs, dim=1)   # [1, num_classes]
        probs_np = probs.cpu().numpy()[0]
        pred_idx = int(np.argmax(probs_np))
        pred_prob = float(probs_np[pred_idx])
    return pred_idx, pred_prob, probs_np

# ===================== 6. SOFT VOTING 3 MODEL =====================
def soft_voting_predict(image_path,
                        vgg_model,
                        res_model,
                        eff_model,
                        weights=(1.0, 1.0, 1.0)):
    """
    Soft voting 3 model dựa trên trung bình vector xác suất.
    weights: (w_vgg, w_resnet, w_effb4)
    """
    # --------- Đọc & tiền xử lý ảnh ---------
    img_pil = Image.open(image_path).convert("RGB")
    x = inference_transform(img_pil).unsqueeze(0).to(device)   # [1, 3, H, W]

    # --------- Dự đoán từng mô hình ---------
    vgg_idx, vgg_prob, vgg_vec = predict_single_model(vgg_model, x)
    res_idx, res_prob, res_vec = predict_single_model(res_model, x)
    eff_idx, eff_prob, eff_vec = predict_single_model(eff_model, x)

    # --------- Soft Voting (trung bình có trọng số) ---------
    w_vgg, w_res, w_eff = weights
    total_w = w_vgg + w_res + w_eff

    avg_probs = (w_vgg * vgg_vec + w_res * res_vec + w_eff * eff_vec) / total_w
    final_idx = int(np.argmax(avg_probs))
    final_prob = float(avg_probs[final_idx])

    # ===================== 7. IN KẾT QUẢ =====================
    print(f"📌 Ảnh đầu vào: {image_path}\n")

    print("🔹 Kết quả từng mô hình:")
    print(f"  - VGG16       → {class_names[vgg_idx]} ({vgg_prob*100:.2f}%)")
    print(f"  - ResNet50    → {class_names[res_idx]} ({res_prob*100:.2f}%)")
    print(f"  - Efficient-B4→ {class_names[eff_idx]} ({eff_prob*100:.2f}%)\n")

    print("🟩 Kết quả Soft Voting (cuối cùng):")
    print(f"  → {class_names[final_idx]}  ({final_prob*100:.2f}%)")

    # Trả về dict nếu muốn dùng tiếp
    return {
        "vgg":       {"idx": vgg_idx, "label": class_names[vgg_idx], "prob": vgg_prob},
        "resnet50":  {"idx": res_idx, "label": class_names[res_idx], "prob": res_prob},
        "effb4":     {"idx": eff_idx, "label": class_names[eff_idx], "prob": eff_prob},
        "soft_vote": {"idx": final_idx, "label": class_names[final_idx], "prob": final_prob}
    }




Using device: cuda
📂 Số lớp (lấy từ folder train): 22
Danh sách lớp: ['Acne', 'Actinic_Keratosis', 'Benign_tumors', 'Bullous', 'Candidiasis', 'DrugEruption', 'Eczema', 'Infestations_Bites', 'Lichen', 'Lupus', 'Moles', 'Psoriasis', 'Rosacea', 'Seborrh_Keratoses', 'SkinCancer', 'Sun_Sunlight_Damage', 'Tinea', 'Unknown_Normal', 'Vascular_Tumors', 'Vasculitis', 'Vitiligo', 'Warts']


C:\Users\letra\AppData\Local\Temp\ipykernel_7956\2321031783.py:63: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(ckpt_path, map_location="cpu")


✅ VGG16 loaded.


C:\Users\letra\AppData\Local\Temp\ipykernel_7956\2321031783.py:83: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(ckpt_path, map_location="cpu")
C:\Users\l

✅ ResNet50 loaded.
Loaded pretrained weights for efficientnet-b4
✅ EfficientNet-B4 loaded.


In [71]:
# ===================== GỌI HÀM =====================
test_image_path = r"E:\archive\SkinDisease\AnhTest\Benign_tumors\dermatofibroma-150.jpeg"

results = soft_voting_predict(
    test_image_path,
    vgg_model=vgg16_model,
    res_model=resnet50_model,
    eff_model=effb4_model,
    weights=(1.0, 1.0, 1.0)   # hoặc (1, 1, 2) nếu muốn ưu tiên Eff-B4
)

📌 Ảnh đầu vào: E:\archive\SkinDisease\AnhTest\Benign_tumors\dermatofibroma-150.jpeg

🔹 Kết quả từng mô hình:
  - VGG16       → Lichen (53.23%)
  - ResNet50    → Benign_tumors (98.52%)
  - Efficient-B4→ Benign_tumors (99.99%)

🟩 Kết quả Soft Voting (cuối cùng):
  → Benign_tumors  (80.83%)


In [69]:
import os

# ===================== 8. SOFT VOTING CHO CẢ FOLDER ẢNH =====================
def soft_voting_predict_folder(folder_path,
                               vgg_model,
                               res_model,
                               eff_model,
                               weights=(1.0, 1.0, 1.0)):
    """
    Đọc toàn bộ ảnh trong 1 folder và in kết quả soft voting cho TỪNG ẢNH.
    """

    exts = (".jpg", ".jpeg", ".png", ".bmp", ".gif")  # các đuôi ảnh cơ bản
    files = [f for f in os.listdir(folder_path)
             if f.lower().endswith(exts)]

    files.sort()  # cho gọn gàng

    if not files:
        print(f"⚠️ Folder không có ảnh: {folder_path}")
        return

    print(f"📁 Đang dự đoán cho folder: {folder_path}")
    print(f"➡️ Số ảnh tìm thấy: {len(files)}\n")

    # Nếu muốn tính độ chính xác theo label của tên folder:
    true_label_name = os.path.basename(folder_path.rstrip("\\/"))
    total = 0
    correct = 0

    for fname in files:
        img_path = os.path.join(folder_path, fname)
        print("===========================================")
        print(f"🖼️ File: {fname}")

        results = soft_voting_predict(
            img_path,
            vgg_model=vgg_model,
            res_model=res_model,
            eff_model=eff_model,
            weights=weights
        )

        # So sánh với "label thật" lấy từ tên folder (nếu trùng class_names)
        pred_label = results["soft_vote"]["label"]
        if pred_label == true_label_name:
            correct += 1
        total += 1

        print()  # dòng trống cho dễ đọc

    # In thống kê cuối cùng (nếu folder là 1 lớp duy nhất)
    if total > 0:
        acc = correct / total * 100
        print("===========================================")
        print(f"✅ Tổng kết folder: {folder_path}")
        print(f"   Label (theo tên folder): {true_label_name}")
        print(f"   Đúng: {correct}/{total}  → Accuracy: {acc:.2f}%")

# ===================== 9. VÍ DỤ GỌI HÀM CHO CẢ FOLDER =====================
test_folder_path = r"E:\archive\SkinDisease\AnhTest\Warts"

soft_voting_predict_folder(
    test_folder_path,
    vgg_model=vgg16_model,
    res_model=resnet50_model,
    eff_model=effb4_model,
    weights=(1.0, 1.0, 1.0)   
)


📁 Đang dự đoán cho folder: E:\archive\SkinDisease\AnhTest\Warts
➡️ Số ảnh tìm thấy: 10

🖼️ File: genital-warts-22.jpeg
📌 Ảnh đầu vào: E:\archive\SkinDisease\AnhTest\Warts\genital-warts-22.jpeg

🔹 Kết quả từng mô hình:
  - VGG16       → Warts (99.32%)
  - ResNet50    → Warts (99.94%)
  - Efficient-B4→ Warts (100.00%)

🟩 Kết quả Soft Voting (cuối cùng):
  → Warts  (99.75%)

🖼️ File: genital-warts-28.jpeg
📌 Ảnh đầu vào: E:\archive\SkinDisease\AnhTest\Warts\genital-warts-28.jpeg

🔹 Kết quả từng mô hình:
  - VGG16       → Warts (100.00%)
  - ResNet50    → Warts (99.15%)
  - Efficient-B4→ Warts (99.95%)

🟩 Kết quả Soft Voting (cuối cùng):
  → Warts  (99.70%)

🖼️ File: genital-warts-30.jpeg
📌 Ảnh đầu vào: E:\archive\SkinDisease\AnhTest\Warts\genital-warts-30.jpeg

🔹 Kết quả từng mô hình:
  - VGG16       → Warts (100.00%)
  - ResNet50    → Warts (99.98%)
  - Efficient-B4→ Warts (99.87%)

🟩 Kết quả Soft Voting (cuối cùng):
  → Warts  (99.95%)

🖼️ File: genital-warts-31.jpeg
📌 Ảnh đầu vào: E:\ar